# 07 — DATA WAREHOUSE & VISUALISASI
### Tabel mart · MySQL · dashboard interaktif

**Peta ke laporan:** Bab 7.1 Proses Membangun Dashboard · Bab 7.2 Interpretasi Bisnis · LO 6

Notebook terakhir. Hasil dari seluruh tahap sebelumnya diringkas menjadi tabel *gold*, ditulis ke **MySQL**, lalu divisualisasikan.

### Kenapa MySQL, bukan membaca Parquet langsung

Instruksi tugas meminta data disimpan pada sistem basis data (`mysql / sql server / postgres`). Lebih dari itu, ada alasan teknis yang nyata: dashboard yang menarik jutaan baris Parquet lewat Spark akan terasa lambat justru ketika dosen mencoba interaktivitasnya. Tabel mart sengaja dibuat **kecil dan sudah teragregasi**, sehingga kueri dashboard selesai dalam milidetik.

MySQL 8.0.46 berjalan di Windows, di luar klaster. Dari dalam container, alamatnya `host.docker.internal` — `bda_common` menukarnya otomatis saat mode klaster, jadi `bda_secret.json` cukup diisi `127.0.0.1`.

### Tiga insight yang disyaratkan

| Insight | Sumber | Isi |
|---|---|---|
| 1 — deskriptif spasial | notebook 02 | Sebaran subtipe menurut negara, wilayah, dan tahun |
| 2 — **machine learning** | notebook 05 | Kinerja klasifikasi, label yang berhasil dipulihkan, selisih HA/NA versus gen internal |
| 3 — **graph analytics** | notebook 06 | Perantara reassortment dan negara paling sentral |

Insight 2 dan 3 wajib ada menurut instruksi, dan keduanya terpenuhi.

---
**Masukan** `stage/sequences` · `models/*` · `graph/*`
**Keluaran** tabel di MySQL `bda_influenza` · `output/dashboard.html` · `output/*.png`

## Bootstrap dan koneksi MySQL

In [ ]:
import sys
sys.path.insert(0, r"D:\BDA\nb" if sys.platform == "win32" else "/workspace/nb")
from bda_common import *

import pandas as pd
import numpy as np
from pyspark.sql import functions as F

info_mesin()
# Notebook ini berjalan DI ATAS YARN, bukan Spark Standalone.
#
# Bisa begitu karena seluruh transformasinya Spark SQL asli, tanpa satu pun
# UDF Python. Executor cukup menjalankan JVM, sehingga NodeManager yang
# berbasis CentOS 7 dan tidak membawa Python sama sekali tetap sanggup
# memprosesnya. Notebook 04, 05, dan 06 tidak bisa begini: ketiganya memakai
# F.udf atau .rdd, yang menuntut proses Python di setiap executor.
#
# Penulisan ke MySQL terjadi DI EXECUTOR, bukan di driver. Dua syaratnya
# sudah diperiksa dan terpenuhi: konektor JDBC berada di $SPARK_HOME/jars
# sehingga ikut terkirim ke YARN bersama pustaka Spark, dan alamat
# host.docker.internal:3306 terjangkau dari dalam container NodeManager.
if info_yarn(diam=True) is None:
    raise RuntimeError(
        f"ResourceManager tidak terjangkau di {YARN_RM}.\n"
        "Notebook ini menuntut profil yarn:\n"
        "    cd D:\\BDA\\docker\n"
        "    docker compose --profile standalone down\n"
        "    docker compose --profile yarn up -d")

spark = spark_session("07-mart-dashboard", master="yarn", konfig={
    "spark.executor.instances": "3",
    "spark.executor.memory": "3g",
    "spark.executor.cores": "2",
    "spark.yarn.am.memory": "1g",
    # Driver berada di container jupyter; executor di NodeManager harus bisa
    # menghubunginya balik lewat nama host itu.
    "spark.driver.host": os.environ.get("HOSTNAME", "jupyter"),
})
print("  penjadwal :", spark.sparkContext.master)
print("  appId     :", spark.sparkContext.applicationId)

RAHASIA = muat_secret()
JDBC_URL, JDBC_PROP = mysql_jdbc(RAHASIA)
print(f"\n  MySQL : {RAHASIA['mysql']['host']}:{RAHASIA['mysql']['port']}"
      f"/{RAHASIA['mysql']['database']}")
print(f"  JDBC  : {JDBC_URL.split('?')[0]}")

In [ ]:
with Tahap("uji koneksi MySQL", "WAREHOUSE"):
    uji = (spark.read.format("jdbc")
           .option("url", JDBC_URL)
           .option("query", "SELECT VERSION() AS versi, DATABASE() AS db, NOW() AS waktu")
           .options(**JDBC_PROP).load())
    uji.show(truncate=False)
    print("  Koneksi berhasil. Konektor JDBC MySQL tersedia di image.")

---
## Membangun tabel mart

Skema bintang sederhana: satu tabel fakta teragregasi ditambah beberapa tabel hasil analisis. Semuanya berukuran kecil karena sudah diringkas — itu memang tujuannya.

In [ ]:
def tulis_mysql(df, nama_tabel, mode="overwrite"):
    """Tulis DataFrame Spark ke MySQL, lalu laporkan jumlah barisnya."""
    n = df.count()
    (df.write.mode(mode).format("jdbc")
       .option("url", JDBC_URL)
       .option("dbtable", nama_tabel)
       .option("batchsize", 5000)
       .options(**JDBC_PROP).save())
    print(f"  {nama_tabel:<28} {n:>8,} baris")
    return n


with Tahap("baca hasil seluruh tahap", "WAREHOUSE"):
    silver = spark.read.parquet(jalur("stage/sequences"))
    print(f"  silver         : {silver.count():,} baris")

    def coba_baca(p, nama):
        try:
            d = spark.read.parquet(p)
            print(f"  {nama:<14} : {d.count():,} baris")
            return d
        except Exception:
            print(f"  {nama:<14} : belum ada (notebook terkait belum dijalankan)")
            return None

    sentralitas = coba_baca(jalur("graph/metrik_sentralitas"), "sentralitas")
    geo_sentral = coba_baca(jalur("graph/geo_sentralitas"), "geo sentral")
    geo_tepi = coba_baca(jalur("graph/geo_tepi"), "geo tepi")
    kandidat = coba_baca(jalur("models/kandidat_reassortant"), "kandidat")
    imputasi = coba_baca(jalur("stage/segmen_terimputasi"), "imputasi")

In [ ]:
with Tahap("bangun dan tulis tabel mart", "WAREHOUSE"):
    # ---------- FAKTA: agregat sekuens ----------
    fakta = (silver
             .filter(F.col("segmen").isNotNull())
             .groupBy("negara", "wilayah", "tahun_koleksi", "segmen", "subtipe", "inang")
             .agg(F.count("*").alias("jumlah_sekuens"),
                  F.round(F.avg("panjang"), 1).alias("panjang_rata2"),
                  F.round(F.avg("gc_pct"), 2).alias("gc_rata2"))
             .withColumnRenamed("tahun_koleksi", "tahun"))
    tulis_mysql(fakta, "fact_sekuens")

    # ---------- DIMENSI ----------
    dim_seg = spark.createDataFrame(
        [(k, v[0], v[1]) for k, v in SEGMEN.items()],
        ["segmen", "gen", "nama_lengkap"]) \
        .withColumn("kelompok", F.when(F.col("segmen").isin(SEG_EKSTERNAL),
                                       F.lit("eksternal (HA/NA)"))
                                 .otherwise(F.lit("internal")))
    tulis_mysql(dim_seg, "dim_segmen")

    dim_sub = (silver.filter(F.col("subtipe").isNotNull())
               .groupBy("subtipe").agg(F.count("*").alias("jumlah_sekuens"),
                                       F.min("tahun_koleksi").alias("tahun_pertama"),
                                       F.max("tahun_koleksi").alias("tahun_terakhir"),
                                       F.countDistinct("negara").alias("jumlah_negara"))
               .withColumn("H", F.regexp_extract("subtipe", r"H(\d+)", 1).cast("int"))
               .withColumn("N", F.regexp_extract("subtipe", r"N(\d+)", 1).cast("int")))
    tulis_mysql(dim_sub, "dim_subtipe")

    dim_neg = (silver.filter(F.col("negara").isNotNull())
               .groupBy("negara", "wilayah")
               .agg(F.count("*").alias("jumlah_sekuens"),
                    F.countDistinct("subtipe").alias("jumlah_subtipe"),
                    F.min("tahun_koleksi").alias("tahun_pertama"),
                    F.max("tahun_koleksi").alias("tahun_terakhir")))
    tulis_mysql(dim_neg, "dim_negara")

    # ---------- KUALITAS DATA (Bab 3.4) ----------
    lap = json.loads((BASE / "stage" / "laporan_kualitas.json").read_text(encoding="utf-8"))
    dq = spark.createDataFrame(pd.DataFrame(lap["aturan"]))
    tulis_mysql(dq, "mart_kualitas_data")

    asal = spark.createDataFrame(pd.DataFrame(
        [{"jenis": "segmen", "sumber": k, "jumlah": v} for k, v in lap["asal_segmen"].items()] +
        [{"jenis": "subtipe", "sumber": k, "jumlah": v} for k, v in lap["asal_subtipe"].items()]))
    tulis_mysql(asal, "mart_asal_label")

In [ ]:
with Tahap("tulis hasil ML dan graf ke MySQL", "WAREHOUSE"):
    # ---------- METRIK MODEL (insight 2) ----------
    p_metrik = BASE / "models" / "metrik_model.csv"
    if p_metrik.exists():
        tulis_mysql(spark.createDataFrame(pd.read_csv(p_metrik)), "mart_metrik_model")
    p_skala = BASE / "models" / "metrik_skalabilitas.csv"
    if p_skala.exists():
        d = pd.read_csv(p_skala)
        d = d.where(pd.notnull(d), None)
        tulis_mysql(spark.createDataFrame(d.astype(str)), "mart_skalabilitas")

    # ---------- SENTRALITAS GRAF (insight 3) ----------
    if sentralitas is not None:
        tulis_mysql(sentralitas.orderBy(F.desc("skor")).limit(500),
                    "mart_sentralitas_lineage")
    if geo_sentral is not None:
        tulis_mysql(geo_sentral, "mart_geo_sentralitas")
    if geo_tepi is not None:
        tulis_mysql(geo_tepi.orderBy(F.desc("lineage_bersama")).limit(2000),
                    "mart_geo_tepi")
    if kandidat is not None:
        tulis_mysql(kandidat.groupBy("subtipe_kelas", "prediksi_subtipe", "negara")
                            .agg(F.count("*").alias("jumlah"),
                                 F.round(F.avg("keyakinan"), 4).alias("keyakinan_rata2")),
                    "mart_kandidat_reassortant")
    if imputasi is not None:
        tulis_mysql(imputasi.groupBy("segmen_prediksi")
                            .agg(F.count("*").alias("jumlah"),
                                 F.round(F.avg("keyakinan"), 4).alias("keyakinan_rata2")),
                    "mart_imputasi_segmen")

In [ ]:
with Tahap("verifikasi isi MySQL", "WAREHOUSE"):
    tabel = (spark.read.format("jdbc").option("url", JDBC_URL)
             .option("query",
                     "SELECT TABLE_NAME AS tabel, TABLE_ROWS AS perkiraan_baris, "
                     "ROUND(DATA_LENGTH/1024,1) AS kb "
                     "FROM information_schema.TABLES "
                     f"WHERE TABLE_SCHEMA = '{RAHASIA['mysql']['database']}' "
                     "ORDER BY TABLE_NAME")
             .options(**JDBC_PROP).load())
    tabel.show(30, truncate=False)
    print("  Seluruh tabel mart dapat dikueri langsung dari MySQL Workbench,")
    print("  Looker Studio, Power BI, atau Tableau.")

---
# INSIGHT 1 — Sebaran subtipe menurut geografi dan waktu

Insight deskriptif spasial. Yang dicari: negara mana menyumbang keragaman terbanyak, subtipe mana mendominasi di wilayah mana, dan bagaimana komposisinya bergeser dari tahun ke tahun.

**Interpretasi bisnis.** Komposisi subtipe per wilayah menentukan stok alat diagnostik cepat dan antivirus yang perlu disiapkan menjelang musim. Belahan bumi utara dan selatan punya musim influenza yang berlawanan, sehingga formulasi dan waktunya pun berbeda.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

with Tahap("INSIGHT 1 -- geografi dan waktu", "VISUALISASI"):
    top_neg = (silver.filter(F.col("negara").isNotNull())
               .groupBy("negara").count().orderBy(F.desc("count")).limit(15).toPandas())
    top_sub = (silver.filter(F.col("subtipe").isNotNull())
               .groupBy("subtipe").count().orderBy(F.desc("count")).limit(12).toPandas())
    per_tahun = (silver.filter(F.col("tahun_koleksi").isNotNull()
                               & F.col("subtipe").isNotNull())
                 .groupBy("tahun_koleksi", "subtipe").count()
                 .orderBy("tahun_koleksi").toPandas())

    fig, ax = plt.subplots(2, 2, figsize=(15, 10))

    ax[0, 0].barh(top_neg["negara"][::-1], top_neg["count"][::-1], color="#0B6E5F")
    ax[0, 0].set_title("15 negara penyumbang sekuens terbanyak")
    ax[0, 0].set_xlabel("jumlah sekuens")

    ax[0, 1].barh(top_sub["subtipe"][::-1], top_sub["count"][::-1], color="#8E5417")
    ax[0, 1].set_title("12 subtipe terbanyak")
    ax[0, 1].set_xlabel("jumlah sekuens")

    utama = top_sub["subtipe"].head(6).tolist()
    piv = (per_tahun[per_tahun["subtipe"].isin(utama)]
           .pivot_table(index="tahun_koleksi", columns="subtipe",
                        values="count", aggfunc="sum").fillna(0))
    piv.plot(ax=ax[1, 0], linewidth=2)
    ax[1, 0].set_title("Pergeseran komposisi subtipe dari tahun ke tahun")
    ax[1, 0].set_xlabel("tahun koleksi"); ax[1, 0].set_ylabel("jumlah sekuens")
    ax[1, 0].legend(fontsize=8)

    wil = (silver.filter(F.col("wilayah").isNotNull())
           .groupBy("wilayah").count().orderBy(F.desc("count")).toPandas())
    ax[1, 1].pie(wil["count"], labels=wil["wilayah"], autopct="%1.1f%%", startangle=90)
    ax[1, 1].set_title("Proporsi sekuens menurut wilayah")

    for a in ax.flat:
        a.grid(alpha=.25)
    plt.tight_layout()
    plt.savefig(OUT / "insight1_geografi.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\n  Negara penyumbang terbesar : {top_neg.iloc[0]['negara']} "
          f"({100*top_neg.iloc[0]['count']/silver.count():.1f}% dari seluruh arsip)")
    print("  Ketimpangan ini WAJIB dibahas sebagai limitasi di Bab 5.6:")
    print("  model yang tidak ditimbang akan mempelajari siapa yang paling rajin")
    print("  menyekuens, bukan biologi influenza.")

---
# INSIGHT 2 — Hasil machine learning

**Wajib menurut instruksi.** Tiga hal yang dilaporkan: kinerja klasifikasi, jumlah label yang berhasil dipulihkan, dan selisih antara model HA/NA dengan model gen internal.

**Interpretasi bisnis.** Klasifikasi segmen otomatis berarti sekuens yang metadatanya tidak lengkap tetap dapat dipakai, tanpa kurasi manual. Untuk laboratorium yang menerima ribuan sekuens per minggu, ini selisih antara data yang terpakai dan data yang menganggur.

Selisih antara model HA/NA dan model gen internal punya makna lebih dalam: ia mengukur seberapa sering segmen influenza bertukar pasangan. Makin besar selisihnya, makin sering *reassortment* terjadi — dan *reassortment* itulah mekanisme yang melahirkan galur pandemi.

In [ ]:
with Tahap("INSIGHT 2 -- machine learning", "VISUALISASI"):
    p = BASE / "models" / "metrik_model.csv"
    if not p.exists():
        print("  Jalankan notebook 05 dulu.")
    else:
        m = pd.read_csv(p)
        display(m)

        fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))

        piv = m.pivot_table(index="algoritma", columns="tugas",
                            values="f1_makro", aggfunc="max")
        piv.plot(kind="bar", ax=ax[0], rot=20)
        ax[0].set_title("F1 makro per algoritma dan tugas")
        ax[0].set_ylabel("F1 makro"); ax[0].set_ylim(0, 1.02)
        ax[0].legend(fontsize=8)

        per_tugas = m.groupby("tugas")["f1_makro"].max()
        warna = ["#0B6E5F" if "A:" in t else ("#8E5417" if "B1" in t else "#93362B")
                 for t in per_tugas.index]
        ax[1].bar(range(len(per_tugas)), per_tugas.values, color=warna)
        ax[1].set_xticks(range(len(per_tugas)))
        ax[1].set_xticklabels([t.split(":")[0] for t in per_tugas.index])
        ax[1].set_title("F1 terbaik per tugas"); ax[1].set_ylim(0, 1.02)
        for i, v in enumerate(per_tugas.values):
            ax[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

        ax[2].bar(m["algoritma"] + "\n" + m["tugas"].str[:2],
                  m["detik_latih"], color="#5A6B65")
        ax[2].set_title("Waktu latih (detik)"); ax[2].tick_params(labelsize=7, rotation=45)

        for a in ax:
            a.grid(alpha=.25, axis="y")
        plt.tight_layout()
        plt.savefig(OUT / "insight2_ml.png", dpi=150, bbox_inches="tight")
        plt.show()

        rp = BASE / "models" / "ringkasan_ml.json"
        if rp.exists():
            r = json.loads(rp.read_text(encoding="utf-8"))
            print(f"\n  F1 subtipe dari HA+NA          : {r['tugas_B1_f1_terbaik']:.4f}")
            print(f"  F1 subtipe dari gen internal   : {r['tugas_B2_f1_terbaik']:.4f}")
            print(f"  Selisih                        : {r['selisih_B1_B2']:.4f}")
            if r["selisih_B1_B2"] > 0.15:
                print("\n  Selisih besar -> gen internal sering berpasangan dengan HA/NA")
                print("  yang berbeda-beda. Ini tanda reassortment yang aktif, dan")
                print("  membenarkan pemodelan influenza sebagai jaringan, bukan pohon.")
            else:
                print("\n  Selisih kecil -> kedelapan segmen cenderung diwariskan")
                print("  sebagai satu paket utuh pada data ini.")

---
# INSIGHT 3 — Graph analytics

**Wajib menurut instruksi.** Dua keluaran: garis keturunan segmen yang menjadi perantara *reassortment*, dan negara yang paling sentral pada jaringan penyebaran.

**Interpretasi bisnis.** Peringkat sentralitas menjawab pertanyaan anggaran yang paling langsung: dengan dana surveilans terbatas, di mana titik pemantauan sebaiknya ditempatkan. Kurva cakupan kumulatif memungkinkan pernyataan berbentuk "K lokasi teratas sudah mencakup P% jalur penyebaran" — bentuk pembenaran anggaran yang bisa langsung dipakai pengambil keputusan.

In [ ]:
with Tahap("INSIGHT 3 -- graph analytics", "VISUALISASI"):
    if sentralitas is None and geo_sentral is None:
        print("  Jalankan notebook 06 dulu.")
    else:
        fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))

        if sentralitas is not None:
            s = sentralitas.orderBy(F.desc("skor")).limit(15).toPandas()
            ax[0].barh(s["id"][::-1], s["skor"][::-1], color="#0B6E5F")
            ax[0].set_title("15 garis keturunan segmen paling sentral\n(perantara reassortment)")
            ax[0].set_xlabel("PageRank"); ax[0].tick_params(labelsize=7)

        if geo_sentral is not None:
            g = geo_sentral.orderBy(F.desc("skor")).limit(15).toPandas()
            ax[1].barh(g["id"][::-1], g["skor"][::-1], color="#8E5417")
            ax[1].set_title("15 negara paling sentral\npada jaringan penyebaran")
            ax[1].set_xlabel("PageRank"); ax[1].tick_params(labelsize=8)

            gg = geo_sentral.orderBy(F.desc("skor")).toPandas()
            kum = 100 * gg["skor"].cumsum() / gg["skor"].sum()
            ax[2].plot(range(1, len(kum) + 1), kum, marker="o", ms=3, color="#93362B")
            ax[2].axhline(80, ls="--", c="gray", lw=1)
            n80 = int((kum < 80).sum()) + 1
            ax[2].axvline(n80, ls="--", c="gray", lw=1)
            ax[2].set_title(f"Cakupan kumulatif\n{n80} negara teratas menutupi 80% jalur")
            ax[2].set_xlabel("jumlah negara (diurutkan)"); ax[2].set_ylabel("cakupan %")

        for a in ax:
            a.grid(alpha=.25)
        plt.tight_layout()
        plt.savefig(OUT / "insight3_graf.png", dpi=150, bbox_inches="tight")
        plt.show()

        if geo_sentral is not None:
            print(f"\n  PERNYATAAN UNTUK LAPORAN:")
            print(f"  \"{n80} negara teratas menurut sentralitas sudah mencakup 80%")
            print(f"   jalur penyebaran garis keturunan influenza. Menempatkan titik")
            print(f"   surveilans hanya di {n80} lokasi tersebut mempertahankan sebagian")
            print(f"   besar daya deteksi dengan sebagian kecil biaya.\"")

---
## Dashboard interaktif

Dibangun sebagai satu berkas HTML mandiri dengan Plotly. Dipilih begini karena bisa dibuka langsung dari berkas tanpa server, tanpa akun, dan tanpa koneksi — dosen cukup membuka berkasnya.

Bila ingin memenuhi anjuran instruksi untuk **mengumpulkan tautan**, tabel mart di MySQL sudah siap: sambungkan Looker Studio atau Power BI ke `bda_influenza`, karena seluruh tabel sudah kecil dan teragregasi.

In [ ]:
with Tahap("bangun dashboard interaktif", "VISUALISASI"):
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio

    neg = top_neg
    sub = top_sub

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Negara penyumbang sekuens terbanyak",
                        "Subtipe terbanyak",
                        "Pergeseran subtipe menurut tahun",
                        "Negara paling sentral (graph analytics)"),
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "scatter"}, {"type": "bar"}]])

    fig.add_trace(go.Bar(x=neg["count"], y=neg["negara"], orientation="h",
                         marker_color="#0B6E5F", name="sekuens"), row=1, col=1)
    fig.add_trace(go.Bar(x=sub["count"], y=sub["subtipe"], orientation="h",
                         marker_color="#8E5417", name="subtipe"), row=1, col=2)

    for s in sub["subtipe"].head(6):
        d = per_tahun[per_tahun["subtipe"] == s]
        fig.add_trace(go.Scatter(x=d["tahun_koleksi"], y=d["count"],
                                 mode="lines+markers", name=s), row=2, col=1)

    if geo_sentral is not None:
        g = geo_sentral.orderBy(F.desc("skor")).limit(15).toPandas()
        fig.add_trace(go.Bar(x=g["skor"], y=g["id"], orientation="h",
                             marker_color="#93362B", name="PageRank"), row=2, col=2)

    fig.update_layout(
        height=900, showlegend=True,
        title_text="Dashboard Analitik Genomik Influenza A — "
                   f"{silver.count():,} sekuens, {RUN_ID}",
        template="plotly_white")
    fig.update_yaxes(autorange="reversed", row=1, col=1)
    fig.update_yaxes(autorange="reversed", row=1, col=2)
    fig.update_yaxes(autorange="reversed", row=2, col=2)

    berkas = OUT / "dashboard.html"
    pio.write_html(fig, file=str(berkas), include_plotlyjs="cdn", full_html=True)
    print(f"  Dashboard disimpan: {berkas}")
    print("  Buka dari Windows di D:\\BDA\\output\\dashboard.html")
    fig.show()

In [ ]:
with Tahap("ringkasan akhir proyek", "MONITORING"):
    print("=" * 66)
    print("  RINGKASAN PIPELINE")
    print("=" * 66)
    print(f"  Sekuens dalam zona silver : {silver.count():,}")
    for nama, p in [("metrik model", BASE / "models" / "metrik_model.csv"),
                    ("skalabilitas", BASE / "models" / "metrik_skalabilitas.csv"),
                    ("ringkasan graf", BASE / "graph" / "ringkasan_graf.json"),
                    ("kualitas data", BASE / "stage" / "laporan_kualitas.json")]:
        print(f"  {nama:<26} {'ADA' if p.exists() else 'belum'}")
    print("\n  Gambar untuk laporan:")
    for g in sorted(OUT.glob("*.png")):
        print(f"    {g.name}")
    print("\n  Dashboard: output/dashboard.html")
    print("  Tabel mart: MySQL bda_influenza")

display(ringkas_zona())
display(jejak_df())
stop_spark()
print("\nSELESAI. Seluruh pipeline 01-07 telah dijalankan.")